# Day 1 — Occupational Stress: Dataset EDA & Target Confirmation
BrainLag Force Wellness expansion — Smart India Hackathon

**Goal (per plan, Section 11, Day 1):** Upload & inspect the Dryad file, confirm the target plan (tertile split of combined DCS + ERI stress score), before any model training happens (Day 2).

In [1]:
import pandas as pd
import numpy as np

xls = pd.ExcelFile('dryad2.xlsx')
print(xls.sheet_names)
df = pd.read_excel(xls, sheet_name='Variables')
legend = pd.read_excel(xls, sheet_name='Legends')
df.shape

['Variables', 'Legends', 'Foglio3']


(289, 27)

## 1. Confirm this is the dataset the plan expects
Plan (Section 4.1): Garbarino & Magnavita (2016), *Work stress and metabolic syndrome in police officers*, Dryad DOI 10.5061/dryad.kj275, ~290 Italian police officers, DCS + ERI instruments.

The plan flagged that Dryad blocks automated downloads, so column names were **not yet verified** against the live file — that verification is this notebook's job.

In [2]:
needed = ['demandmedia','controlmedia','supportmedia','DCRmedia',
          'effortmedia','rewardmedia','overmedia','ERImedia']
check = pd.DataFrame({
    'present': [c in df.columns for c in needed],
    'min': [df[c].min() if c in df.columns else None for c in needed],
    'max': [df[c].max() if c in df.columns else None for c in needed],
}, index=needed)
check

,present,min,max
demandmedia,True,8.00,20.00
controlmedia,True,6.00,20.00
supportmedia,True,6.00,24.00
DCRmedia,True,0.55,4.00
effortmedia,True,7.67,28.00
rewardmedia,True,16.00,51.67
overmedia,True,6.00,24.00
ERImedia,True,0.32,3.09


**Result: all 8 DCS/ERI columns are present with the exact names the plan expected (`demandmedia`, `controlmedia`, `supportmedia`, `DCRmedia`, `effortmedia`, `rewardmedia`, `overmedia`, `ERImedia`). No renaming or remapping needed for Day 2 training.**

In [3]:
print('Rows:', len(df), ' (plan estimated ~290; file actually has', len(df), '— close enough, no action needed)')
print('Total missing values:', df.isna().sum().sum())
df.isna().sum().to_frame('missing')

Rows: 289  (plan estimated ~290; file actually has 289 — close enough, no action needed)
Total missing values: 0


,missing
eta,0
anzianita_servizio,0
Qual,0
Scol,0
Geo,0
Single,0
House,0
Prole,0
stai,0
bdi,0


**Result: zero missing values across all 27 columns and 289 rows.** No imputation strategy needed before Day 2 training.

In [4]:
df[needed].describe().T

,count,mean,std,min,25%,50%,75%,max
demandmedia,289.0,13.444498,2.028452,8.00,12.00,13.00,15.00,20.00
controlmedia,289.0,13.257370,2.678704,6.00,11.67,13.67,15.00,20.00
supportmedia,289.0,18.584048,2.880755,6.00,17.33,18.67,20.67,24.00
DCRmedia,289.0,1.306263,0.411364,0.55,1.06,1.21,1.44,4.00
effortmedia,289.0,15.038097,3.156132,7.67,12.67,14.67,16.67,28.00
rewardmedia,289.0,42.259654,6.175189,16.00,40.00,44.00,46.33,51.67
overmedia,289.0,6.873218,1.898233,6.00,6.00,6.00,7.00,24.00
ERImedia,289.0,0.702180,0.287017,0.32,0.53,0.63,0.78,3.09


## 2. Build & confirm the target: tertile split of combined DCS + ERI stress
Plan (Section 4.3, primary option): split officers into 3 equal groups by their **combined DCS + ERI stress score**, matching the split style the original study authors used.

`DCRmedia` is already the DCS job-strain ratio (demand/control) and `ERImedia` is already the ERI imbalance ratio (effort/reward) — both provided directly in the file. We combine them by z-scoring each (they're on different scales) and summing, then cut into tertiles.

In [5]:
z_dcr = (df['DCRmedia'] - df['DCRmedia'].mean()) / df['DCRmedia'].std()
z_eri = (df['ERImedia'] - df['ERImedia'].mean()) / df['ERImedia'].std()
df['combined_stress_z'] = z_dcr + z_eri

q1, q2 = df['combined_stress_z'].quantile([1/3, 2/3])

def to_class(v):
    if v <= q1: return 'Low'
    if v <= q2: return 'Moderate'
    return 'High'

df['target_tertile'] = df['combined_stress_z'].apply(to_class)
print('Cut points:', round(q1,3), round(q2,3))
df['target_tertile'].value_counts()

Cut points: -0.879 0.202


target_tertile
Low         97
High        96
Moderate    96
Name: count, dtype: int64

In [6]:
(df['target_tertile'].value_counts(normalize=True) * 100).round(1)

target_tertile
Low         33.6
High        33.2
Moderate    33.2
Name: proportion, dtype: float64

**Result: 33.6% / 33.2% / 33.2% — a clean, balanced 3-way split.** No class-imbalance handling needed for the Day 2 classifier. The primary plan works; the iso-strain fallback (Section 4.3, fallback option) is not needed. We sanity-check it anyway below, since it's an independent, published clinical rule.

In [7]:
med_d, med_c = df['demandmedia'].median(), df['controlmedia'].median()

def iso_strain(row):
    # Published rule: High demand + Low control = High risk ('iso-strain')
    return 'High' if (row['demandmedia'] > med_d and row['controlmedia'] < med_c) else 'Low'

df['iso_strain'] = df.apply(iso_strain, axis=1)
pd.crosstab(df['iso_strain'], df['target_tertile'])

target_tertile,High,Low,Moderate
iso_strain,,,
High,51,0,16
Low,45,97,80


**Sanity check passed:** every officer flagged `High` by the clinical iso-strain rule lands in the tertile target's `Moderate` or `High` group — **zero** land in `Low`. The two independent methods agree directionally, which supports using the tertile target as the Day 2 training label.

## 3. Decision for Day 2
- **Target confirmed:** `target_tertile` (Low / Moderate / High), built from z-scored `DCRmedia + ERImedia`, tertile-split.
- **Not used:** BDI depression flag, MBI burnout scores, metabolic-health variables — none of these substitute for the stress label, per Section 4.3's exclusion rule.
- **Fallback (iso-strain) not required** — primary plan produced a clean, balanced split.
- Saved below for Day 2's `train_occupational_stress.py` to load directly.

In [8]:
df.to_csv('occupational_stress_with_target.csv', index=False)
print('Saved occupational_stress_with_target.csv —', df.shape[0], 'rows,', df.shape[1], 'columns (incl. target_tertile)')

Saved occupational_stress_with_target.csv — 289 rows, 30 columns (incl. target_tertile)
